# ColdSite-DTI — does attention find *this drug's* contacts? (Kaggle, T4 x2)

Both ground truths in the audit so far are per protein: UniProt's annotated residues and
the 85-residue KLIFS ATP pocket. A reviewer's fairest objection is that the model was
asked about a **pair**, so the site that matters is where *that drug* actually binds.

`src/data/klifs_ligand_contacts.py` builds that ground truth from KLIFS interaction
fingerprints — for every co-crystal structure, which of the 85 pocket positions touch the
bound ligand — for the DAVIS drugs that have been crystallised with a DAVIS kinase
(matched by InChIKey). This notebook scores the trained models against it, in three arms:

| arm | ground truth | what it answers |
|---|---|---|
| **drug** | the pair's own contacts | does attention hit the residues this drug touches? |
| **swapped** | another drug's contacts on the same protein | or would any drug's contacts in that pocket have scored the same? |
| **one pair per protein** | the pair's own contacts, first pair only | the conservative unit, in case pairs of one protein are correlated |

The swapped arm is the point. Drug contacts sit inside one pocket, so a model that merely
finds the pocket scores well against *any* drug's contacts there; only the gap between the
first two arms is specific to the drug in the pair.

## What to do

1. **Settings** (right panel): Accelerator **GPU T4 x2**, Internet **On**.
2. **Add Input**: the same dataset(s) of trained cells the analysis notebook uses.
3. **Save Version -> Save & Run All (Commit)**, then download `drug_sites_davis.zip`.

It trains nothing and scores far fewer rows than the main analysis, so it is short —
well under an hour on two T4s for the models whose cells are attached.


## 1. Settings — the only cell you should need to edit

In [ ]:
# ============================================================================
# SETTINGS
# ============================================================================

DATASET = 'davis'

# The audited models: DeepDTA has no attention and nothing to score here.
MODELS = ['coldsite_dti', 'hyperattentiondti', 'moltrans']
SEEDS = [1, 2, 3]

RUN_DRUG = True          # arm 1: each pair against its own drug's contacts
RUN_SWAPPED = True       # arm 2: against another drug's contacts, same protein
RUN_PAIRED = True        # arm 3: arm 1 restricted to the pairs the swap can cover
RUN_ONE_PER_PROTEIN = True   # arm 4: arm 1, one pair per protein
SKIP_EXISTING = True
DEADLINE_HOURS = 11

# The input check in section 5 covers the whole 48-cell grid; a model whose cells are not
# attached is dropped there, and the arms below then skip it.
AUDIT_MODELS = list(MODELS)
ALL_MODELS = ['deepdta', 'coldsite_dti', 'hyperattentiondti', 'moltrans']
RUN_STAGE3 = False       # no audit table here: this notebook only scores the new arms
INPUT_ROOT = '/kaggle/input'

KNOWN = ['deepdta', 'coldsite_dti', 'hyperattentiondti', 'moltrans']
assert DATASET == 'davis', 'the drug-specific ground truth is DAVIS-only so far'
assert MODELS and all(m in KNOWN for m in MODELS), MODELS
assert SEEDS, 'SEEDS must not be empty'
print(f'{DATASET} | models: {", ".join(MODELS)} | seeds {SEEDS}')
print('arms:', ', '.join(n for n, on in (('drug', RUN_DRUG), ('swapped', RUN_SWAPPED),
                                         ('drug (paired)', RUN_PAIRED),
                                         ('one pair per protein', RUN_ONE_PER_PROTEIN)) if on))


## 2. The GPUs and the clock

In [ ]:
import time
START = time.time()              # the self-stop is measured from here

import torch

assert torch.cuda.is_available(), 'No CUDA. Settings -> Accelerator -> GPU.'
N_GPU = torch.cuda.device_count()
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'GPU {i}: {p.name}, {p.total_memory/1e9:.1f} GB')
print('torch  :', torch.__version__)
DEADLINE = START + DEADLINE_HOURS * 3600


def hours_left():
    return (DEADLINE - time.time()) / 3600


print(f'{N_GPU} GPU(s); self-stop in {hours_left():.1f} h')
if N_GPU < 2:
    print('NOTE: one GPU only -- everything still runs, just serially (about twice as long).')


## 3. Clone the repo

The analysis code, the aligned UniProt ground truth, the KLIFS pocket definitions and the
non-kinase panel all live in the repository, so nothing here is fetched by hand.

In [ ]:
import os

REPO = 'https://github.com/Mahim56207/ColdSite-DTI_New.git'
WORK = '/kaggle/working'
SRC = f'{WORK}/ColdSite-DTI_New'

if not os.path.exists(SRC):
    !git clone --branch main {REPO} {SRC}
os.chdir(SRC)
!git pull origin main
!pip install -q tabulate subword-nmt

RESULTS = f'{WORK}/results'                        # the trained cells land here
OUT = f'{WORK}/analysis_{DATASET}_policyA'         # UniProt analysis outputs
OUT_KLIFS = f'{OUT}_klifs'                         # KLIFS ladders (same file names)
for d in (RESULTS, OUT, OUT_KLIFS):
    os.makedirs(d, exist_ok=True)

# The pieces this notebook cannot run without, all of them committed files.
NEEDED = ['src/evaluation/run_all.py', 'src/evaluation/positional_control.py',
          'src/evaluation/clean_accuracy.py', 'src/evaluation/exclusions.py',
          f'data/{DATASET}_ground_truth_sites.json', f'data/{DATASET}_klifs_pocket_sites.json',
          'data/nonkinase_ground_truth_sites.json', 'data/processed/nonkinase_panel.csv']
missing = [p for p in NEEDED if not os.path.exists(p)]
assert not missing, ('this checkout is missing ' + ', '.join(missing) +
                     ' -- push the commit that adds them to origin/main, then re-run this cell')
print()
!git log --oneline -1
print('cells  ->', RESULTS)
print('outputs->', OUT, 'and', OUT_KLIFS)


## 4. The DAVIS files and the splits

The analysis reads test rows from the split files, so they have to exist here and be the
**same** splits everything was trained on. The row counts are checked against the numbers
recorded on three machines; a mismatch stops the notebook.

`load_data` and `build_splits` each handle both DAVIS and KIBA in one pass, so KIBA's
source files are downloaded as well even though nothing here analyses KIBA.

In [ ]:
BASE = 'https://raw.githubusercontent.com/hkmztrk/DeepDTA/master/data'
# Both loaders below loop over ('davis', 'kiba'), so KIBA's files have to be here too
# even when only DAVIS is analysed -- they are small.
for ds in ('davis', 'kiba'):
    for fname in ('ligands_can.txt', 'proteins.txt', 'Y'):
        target = f'src/data/baselines/deepdta/data/{ds}/{fname}'
        os.makedirs(os.path.dirname(target), exist_ok=True)
        if not os.path.exists(target):
            !curl -sL {BASE}/{ds}/{fname} -o {target}
        assert os.path.getsize(target) > 1000, f'{target} did not download'
!python -m src.data.load_data 2>&1 | tail -2
!python -m src.data.build_splits 2>&1 | grep -E 'davis|leakage'

import pandas as pd

EXPECTED_SPLITS = {'random': (21039, 3006, 6011), 'cold_drug': (21658, 2652, 5746),
                   'cold_target': (21080, 2992, 5984), 'cold_pair': (15190, 264, 1144)}
problems = []
for split, expected in EXPECTED_SPLITS.items():
    got = tuple(len(pd.read_csv(f'data/splits/{DATASET}/{split}/{p}.csv'))
                for p in ('train', 'valid', 'test'))
    print(f'{split:12s} {str(got):26s} {"OK" if got == expected else "MISMATCH"}')
    if got != expected:
        problems.append(f'{split}: expected {expected}, got {got}')
assert not problems, ('these splits are not the ones the cells were trained on:\n  '
                      + '\n  '.join(problems))
print('\nsplits match the trained cells.')


## 5. Collect the trained cells — and verify every single one

Every attached dataset is searched for the 48 `(model, level, seed)` cells. For each cell
the **test AUROC in its results file must match the value this project verified on the
Mac**, to four decimals. That one check catches, without any judgement on your part:

- a checkpoint that was cut off mid-training (its results file is absent, or its number
  differs);
- MolTrans's invalid seeds 2 and 3 from the first grid, which trained as seed 1 (the
  vendored `models.py` reseeds torch on import) and whose AUROCs are seed 1's;
- an older copy of a cell that was later retrained;
- a dataset that is simply the wrong one.

If several attached datasets hold the same cell, the copy whose AUROC matches is the one
used, and its checkpoint is taken from the same folder. Nothing is guessed: if no copy
matches, the notebook stops and names the cell.

In [ ]:
import glob
import json
import shutil

LEVELS = ('random', 'cold_drug', 'cold_target', 'cold_pair')
TOLERANCE = 1e-4          # the file is a copy of a verified run, so this is exact equality

# Test AUROC of each verified cell (merged folder ~/ColdSite-results/davis_binary,
# 2026-09-14, after account 1 v3 and MolTrans's retrained seeds 2-3).
EXPECTED_AUROC = {
    ('deepdta', 'random', 1): 0.931653,
    ('deepdta', 'random', 2): 0.927969,
    ('deepdta', 'random', 3): 0.927453,
    ('deepdta', 'cold_drug', 1): 0.737688,
    ('deepdta', 'cold_drug', 2): 0.650683,
    ('deepdta', 'cold_drug', 3): 0.686223,
    ('deepdta', 'cold_target', 1): 0.903944,
    ('deepdta', 'cold_target', 2): 0.910353,
    ('deepdta', 'cold_target', 3): 0.908054,
    ('deepdta', 'cold_pair', 1): 0.768019,
    ('deepdta', 'cold_pair', 2): 0.712526,
    ('deepdta', 'cold_pair', 3): 0.702702,
    ('coldsite_dti', 'random', 1): 0.925436,
    ('coldsite_dti', 'random', 2): 0.923188,
    ('coldsite_dti', 'random', 3): 0.923381,
    ('coldsite_dti', 'cold_drug', 1): 0.723254,
    ('coldsite_dti', 'cold_drug', 2): 0.712099,
    ('coldsite_dti', 'cold_drug', 3): 0.726471,
    ('coldsite_dti', 'cold_target', 1): 0.850112,
    ('coldsite_dti', 'cold_target', 2): 0.851250,
    ('coldsite_dti', 'cold_target', 3): 0.869614,
    ('coldsite_dti', 'cold_pair', 1): 0.737520,
    ('coldsite_dti', 'cold_pair', 2): 0.556798,
    ('coldsite_dti', 'cold_pair', 3): 0.576517,
    ('hyperattentiondti', 'random', 1): 0.940107,
    ('hyperattentiondti', 'random', 2): 0.939488,
    ('hyperattentiondti', 'random', 3): 0.931396,
    ('hyperattentiondti', 'cold_drug', 1): 0.720117,
    ('hyperattentiondti', 'cold_drug', 2): 0.757170,
    ('hyperattentiondti', 'cold_drug', 3): 0.803226,
    ('hyperattentiondti', 'cold_target', 1): 0.915160,
    ('hyperattentiondti', 'cold_target', 2): 0.916101,
    ('hyperattentiondti', 'cold_target', 3): 0.913219,
    ('hyperattentiondti', 'cold_pair', 1): 0.696443,
    ('hyperattentiondti', 'cold_pair', 2): 0.655194,
    ('hyperattentiondti', 'cold_pair', 3): 0.730135,
    ('moltrans', 'random', 1): 0.921960,
    ('moltrans', 'random', 2): 0.925391,
    ('moltrans', 'random', 3): 0.921102,
    ('moltrans', 'cold_drug', 1): 0.667724,
    ('moltrans', 'cold_drug', 2): 0.680167,
    ('moltrans', 'cold_drug', 3): 0.706277,
    ('moltrans', 'cold_target', 1): 0.867992,
    ('moltrans', 'cold_target', 2): 0.878513,
    ('moltrans', 'cold_target', 3): 0.875572,
    ('moltrans', 'cold_pair', 1): 0.589877,
    ('moltrans', 'cold_pair', 2): 0.567391,
    ('moltrans', 'cold_pair', 3): 0.548271,
}


def names(model, level, seed):
    """(results file, checkpoint file) as the project's checkpoint_naming writes them."""
    suffix = '' if model == 'coldsite_dti' else f'_{model}'
    return (f'{DATASET}_{level}_binary_seed{seed}{suffix}_results.json',
            f'coldsite_dti_{DATASET}_{level}_binary_seed{seed}{suffix}.pt')


found = {}
for pattern in ('*_results.json', '*.pt'):
    for path in glob.glob(f'{INPUT_ROOT}/**/{pattern}', recursive=True):
        found.setdefault(os.path.basename(path), []).append(path)

# What is attached and what each dataset holds: the first thing to look at when a cell
# comes up missing.
attached = sorted(os.listdir(INPUT_ROOT)) if os.path.isdir(INPUT_ROOT) else []
print(f'{len(attached)} dataset(s) attached at {INPUT_ROOT}:')
for name in attached:
    root = os.path.join(INPUT_ROOT, name)
    res = len(glob.glob(f'{root}/**/*_results.json', recursive=True))
    ckpt = len(glob.glob(f'{root}/**/*.pt', recursive=True))
    print(f'   {name}: {res} results file(s), {ckpt} checkpoint(s)')
assert found, (
    f'nothing to analyse: no results files and no checkpoints anywhere under '
    f'{INPUT_ROOT}. Use **Add Input** in the right-hand panel to attach the dataset(s) '
    f'holding the trained cells, then re-run. Attached now: {attached or "nothing"}.')
print(f'{len(found)} distinct file name(s) across the attached inputs\n')

# Mandatory: the models this run reports on, plus the audit family when stage 3 runs.
# DeepDTA has no attention -- it appears only in the summary's accuracy table, so its
# cells are welcome but not required.
REQUIRED = sorted(set(MODELS) | (set(AUDIT_MODELS) if RUN_STAGE3 else set()))
print('required:', ', '.join(REQUIRED), '| optional:',
      ', '.join(m for m in ALL_MODELS if m not in REQUIRED) or 'none')

taken, problems, optional_missing = [], [], []
for (model, level, seed), expected in sorted(EXPECTED_AUROC.items()):
    res_name, ckpt_name = names(model, level, seed)
    chosen = None
    for candidate in found.get(res_name, []):
        try:
            auroc = json.load(open(candidate))['test_metrics']['auroc']
        except Exception as exc:
            problems.append(f'{model} {level} s{seed}: unreadable {candidate} ({exc})')
            continue
        if abs(auroc - expected) <= TOLERANCE:
            chosen = candidate
            break
    if chosen is None and model not in REQUIRED:
        optional_missing.append(f'{model} {level} s{seed}')
        continue
    if chosen is None and model in REQUIRED:
        seen = []
        for c in found.get(res_name, []):
            try:
                seen.append(f'{os.path.basename(os.path.dirname(c))} '
                            f'{json.load(open(c))["test_metrics"]["auroc"]:.4f}')
            except Exception:
                seen.append(f'{os.path.basename(os.path.dirname(c))} unreadable')
        problems.append((model, f'{model} {level} s{seed}: expected AUROC {expected:.4f}, '
                         + (f'attached inputs have {"; ".join(seen)}' if seen
                            else 'no results file found')))
        continue
    if chosen is None:
        optional_missing.append(f'{model} {level} s{seed}')
        continue
    ckpt = os.path.join(os.path.dirname(chosen), ckpt_name)
    if not os.path.exists(ckpt):
        message = (f'{model} {level} s{seed}: results file matches but its checkpoint '
                   f'{ckpt_name} is not beside it')
        if model in REQUIRED:
            problems.append((model, message))
        else:
            optional_missing.append(message)
        continue
    for src in (chosen, ckpt):
        dst = os.path.join(RESULTS, os.path.basename(src))
        if not os.path.exists(dst):
            try:                       # MolTrans's 12 checkpoints are ~240 MB each
                os.symlink(src, dst)
            except OSError:
                shutil.copy2(src, dst)
    taken.append((model, level, seed))

print(f'verified and linked: {len(taken)}/48 cells '
      f'({sum(1 for c in taken if c[0] in REQUIRED)} of the {12 * len(REQUIRED)} '
      f'this run needs)')
if optional_missing:
    print(f'not attached, and not needed ({len(optional_missing)}): '
          + ', '.join(optional_missing[:6]) + (' ...' if len(optional_missing) > 6 else ''))
    print('   (DeepDTA is the accuracy anchor only: the summary table will omit it)')
if problems:
    broken = sorted({m for m, _ in problems})
    print(f'\n{"!" * 70}')
    print(f'{len(problems)} cell(s) are NOT the verified ones, so '
          f'{", ".join(broken)} is dropped from this run:')
    for _m, message in problems:
        print('  -', message)
    print('Attach the dataset holding those cells and re-run to include them. Everything '
          'else below is analysed as usual.')
    print('!' * 70)

# A checkpoint with no results file beside it is an unfinished cell. run_all refuses to
# analyse one, and so does this: it would score a half-trained model.
orphans = [os.path.basename(p) for p in glob.glob(f'{RESULTS}/*.pt')
           if not any(os.path.basename(p) == names(m, lv, s)[1] for m, lv, s in taken)]
assert not orphans, f'unfinished cell(s) in {RESULTS}: {orphans}'

# A model is usable by the later stages only with all 12 of its cells.
PRESENT = [m for m in ALL_MODELS if sum(1 for c in taken if c[0] == m) == 12]
MODELS = [m for m in MODELS if m in PRESENT]
AUDIT_MODELS = [m for m in AUDIT_MODELS if m in PRESENT]
ALL_MODELS = [m for m in ALL_MODELS if m in PRESENT]
assert MODELS, ('none of the models in MODELS has all 12 of its cells verified -- '
                'nothing could be analysed. See the list above.')
print('every required cell is the verified one, and no unfinished checkpoints.')
print(f'complete models: {", ".join(PRESENT)}')
print(f'-> per-model reports: {", ".join(MODELS)} | audit: {", ".join(AUDIT_MODELS)}')


## 6. The runner

Each stage is a list of real command-line calls — the same commands the project runs on a
laptop — spread over the GPUs. Output is streamed with a `[GPU0 ...]` prefix so two
parallel jobs stay readable, a STATUS line prints every 10 minutes, and everything stops at
the deadline with its finished outputs intact.

In [ ]:
import re
import subprocess
import threading

STATUS_EVERY = 600


def py(module, *args):
    return ['python', '-u', '-m', module, *map(str, args)]


def run_parallel(queues, label):
    """queues: {gpu: [(name, cmd, expected_output), ...]}. True if the deadline cut in."""
    queues = {g: q for g, q in queues.items() if q}
    where = {g: {'name': '-', 'done': 0, 'total': len(q)} for g, q in queues.items()}
    state = {'deadline': False, 'failed': []}
    running = {}

    def worker(gpu, jobs):
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu), 'PYTHONUNBUFFERED': '1'}
        with open(f'{WORK}/{label}_gpu{gpu}.log', 'a') as log:
            for name, cmd, expected in jobs:
                if time.time() > DEADLINE:
                    return
                if SKIP_EXISTING and expected and os.path.exists(expected):
                    print(f'[skip] GPU{gpu} {name} -- {os.path.basename(expected)} exists',
                          flush=True)
                    where[gpu]['done'] += 1
                    continue
                where[gpu]['name'] = name
                began = time.time()
                print(f'\n{"=" * 70}\n[GPU{gpu}] {name}\n{"=" * 70}', flush=True)
                proc = subprocess.Popen(cmd, env=env, stdout=subprocess.PIPE,
                                        stderr=subprocess.STDOUT, text=True, bufsize=1)
                running[gpu] = proc
                for line in proc.stdout:
                    if not re.match(r'\s*(test |train )?batch \d+/', line):   # drop batch spam
                        print(f'[GPU{gpu}] {line}', end='', flush=True)
                    log.write(line)
                code_ = proc.wait()
                where[gpu]['done'] += 1
                took = time.time() - began
                if code_ != 0 or (expected and not os.path.exists(expected)):
                    why = f'exited {code_}' if code_ else f'wrote no {os.path.basename(expected)}'
                    print(f'!! [GPU{gpu}] {name} FAILED ({why}, {took:.0f}s)', flush=True)
                    if not state['deadline']:
                        state['failed'].append(name)
                else:
                    print(f'   [GPU{gpu}] {name} done in {took:.0f}s', flush=True)

    threads = [threading.Thread(target=worker, args=(g, q), daemon=True)
               for g, q in queues.items()]
    for t in threads:
        t.start()
    last = 0.0
    while any(t.is_alive() for t in threads):
        if time.time() - last >= STATUS_EVERY:
            last = time.time()
            parts = [f'GPU{g}: {w["name"]} ({w["done"]}/{w["total"]})' for g, w in where.items()]
            print(f'\n=== STATUS {time.strftime("%H:%M")} | ' + ' | '.join(parts)
                  + f' | {hours_left():.1f} h left ===\n', flush=True)
        if time.time() > DEADLINE and not state['deadline']:
            state['deadline'] = True
            print('\n*** deadline: stopping so this commit saves its output. Re-run the '
                  'notebook with the output attached to finish the rest. ***\n', flush=True)
            for proc in list(running.values()):
                if proc.poll() is None:
                    proc.terminate()
        time.sleep(10)
    if state['failed']:
        print('\nFAILED: ' + ', '.join(state['failed']))
    return state['deadline']


def spread(jobs):
    """Deal jobs to the GPUs, heaviest first, so both finish at about the same time."""
    return {g: jobs[g::N_GPU] for g in range(N_GPU)}


COMMON = ['--split-root', 'data/splits', '--checkpoint-dir', RESULTS]
GT = f'data/{DATASET}_ground_truth_sites.json'
GT_KLIFS = f'data/{DATASET}_klifs_pocket_sites.json'
print('runner ready')


## 7. The drug-specific ground truth, and how much of each split it covers

The two files come from the repository (built by `src/data/klifs_ligand_contacts.py`, which
needs the KLIFS API and RDKit, so it is run once on a laptop rather than here). This cell
prints, per split, how many test pairs can be scored at all — the honest limit on this
measurement, and the first thing to read in the output.

In [ ]:
import json
import pandas as pd

GT_DRUG = f'data/{DATASET}_drug_sites.json'
GT_SWAP = f'data/{DATASET}_drug_sites_swapped.json'
GT_PAIRED = f'data/{DATASET}_drug_sites_paired.json'
for path in (GT_DRUG, GT_SWAP, GT_PAIRED):
    assert os.path.exists(path), (
        f'{path} is not in this checkout -- run "python -m src.data.klifs_ligand_contacts '
        f'--dataset {DATASET}" and push it, then re-run section 3')

drug_sites = json.load(open(GT_DRUG))
swapped = json.load(open(GT_SWAP))
pairs = {(k.split('|')[0], k.split('|')[1]) for k in drug_sites}
print(f'{len(drug_sites)} crystallised pairs '
      f'({len({t for _d, t in pairs})} proteins, {len({d for d, _t in pairs})} drugs); '
      f'{len(swapped)} of them can also be swapped\n')

LEVELS = ('random', 'cold_drug', 'cold_target', 'cold_pair')
print(f'{"level":12s} {"test rows":>10s} {"scorable":>9s} {"proteins":>9s} {"swapped":>8s}')
COVERAGE = {}
for level in LEVELS:
    frame = pd.read_csv(f'data/splits/{DATASET}/{level}/test.csv')
    keys = [f'{d}|{t}' for d, t in zip(frame.Drug_ID.astype(str), frame.Target_ID.astype(str))]
    mine = [k for k in keys if k in drug_sites]
    COVERAGE[level] = len(mine)
    print(f'{level:12s} {len(frame):10d} {len(mine):9d} '
          f'{len({k.split("|")[1] for k in mine}):9d} '
          f'{sum(1 for k in mine if k in swapped):8d}')
print('\nA level with no scorable pair is skipped below -- with 13 held-out drugs, the '
      'cold-drug and cold-pair levels can be thin or empty, and that is a result about '
      'what DAVIS can support, not a failure.')


## 8. Score the arms

`run_ladder` reads either kind of ground truth: `src/data/ground_truth.py`'s `site_lookup`
keys by protein for the UniProt and KLIFS files and by `"<drug>|<protein>"` for these, so
the arms differ only in the `--ground-truth` file (and, for arm 3, the unit of averaging).
`--pairs-per-target 0` keeps every scorable pair; the default of 1 keeps the first pair of
each protein.

In [ ]:
ARMS = []
if RUN_DRUG:
    ARMS.append(('drug', GT_DRUG, 0, f'{WORK}/drug_sites_{DATASET}'))
if RUN_SWAPPED:
    ARMS.append(('swapped', GT_SWAP, 0, f'{WORK}/drug_sites_{DATASET}_swapped'))
if RUN_PAIRED:
    # the drug arm on exactly the swapped arm's keys: same pairs, same n, so the only
    # difference between those two columns is WHICH drug the contacts belong to
    ARMS.append(('drug (paired)', GT_PAIRED, 0, f'{WORK}/drug_sites_{DATASET}_paired'))
if RUN_ONE_PER_PROTEIN:
    ARMS.append(('one pair per protein', GT_DRUG, 1, f'{WORK}/drug_sites_{DATASET}_perprotein'))
for _name, _gt, _cap, folder in ARMS:
    os.makedirs(folder, exist_ok=True)

jobs = []
for name, gt, cap, folder in ARMS:
    for m in MODELS:
        tag = f'{m}_{DATASET}' if m != 'coldsite_dti' else DATASET
        for s in SEEDS:
            jobs.append((f'{m} seed {s}: {name} arm',
                         py('src.evaluation.run_ladder', '--model', m, '--task', 'binary',
                            '--dataset', DATASET, '--seed', s, *COMMON,
                            '--ground-truth', gt, '--pairs-per-target', cap,
                            '--out-dir', folder, '--device', 'cuda'),
                         os.path.join(folder, f'ladder_{tag}_seed{s}.json')))

if jobs:
    cut = run_parallel(spread(jobs), 'drugsites')
    print('\n' + ('CUT SHORT by the deadline' if cut else 'every arm finished'))
else:
    print('Nothing to run: every arm is switched off in the settings cell.')


## 9. The comparison

precision@10 in each arm, beside its own chance level (which differs: a drug touches about
19 residues, the whole pocket 85). The number that answers the question is
**drug − swapped**: the part of the agreement that is specific to the drug in the pair,
rather than to the pocket it binds in.

In [ ]:
import statistics as st

K = '10'          # the k the paper reports


def read(folder, model, seed):
    tag = f'{model}_{DATASET}' if model != 'coldsite_dti' else DATASET
    path = os.path.join(folder, f'ladder_{tag}_seed{seed}.json')
    return json.load(open(path)) if os.path.exists(path) else None


def at_k(payload, level, field='precision_at_k'):
    """One number out of a ladder JSON: {level: {by_k: {k: {...}}}}."""
    if not payload or level not in payload:
        return None
    return payload[level].get('by_k', {}).get(K, {}).get(field)


def mean_sd(values):
    """Mean and sd of the seeds, or an em dash.

    Named so it cannot shadow the runner cell's spread(), which deals jobs to GPUs, and
    computed with plain arithmetic: statistics.stdev raises AttributeError on some value
    types under the Python 3.12 that Kaggle runs, which killed this cell after every arm
    had already finished.
    """
    xs = [float(v) for v in values if v is not None]
    if not xs:
        return '—'
    mean = sum(xs) / len(xs)
    if len(xs) < 2:
        return f'{mean:.3f}'
    sd = (sum((x - mean) ** 2 for x in xs) / (len(xs) - 1)) ** 0.5
    return f'{mean:.3f} ± {sd:.3f}'


rows = []
for m in MODELS:
    for level in LEVELS:
        line = {'model': m, 'level': level}
        means = {}
        for name, _gt, _cap, folder in ARMS:
            payloads = [read(folder, m, s) for s in SEEDS]
            values = [at_k(p, level) for p in payloads]
            line[name] = mean_sd(values)
            kept = [v for v in values if v is not None]
            means[name] = sum(map(float, kept)) / len(kept) if kept else None
            if name == 'drug':
                line['chance'] = mean_sd([at_k(p, level, 'chance') for p in payloads])
                pairs = [at_k(p, level, 'n_evaluated') for p in payloads]
                line['n'] = max([x for x in pairs if x is not None], default='—')
        if means.get('drug') is not None and means.get('swapped') is not None:
            line['drug - swapped'] = f"{means['drug'] - means['swapped']:+.3f}"
        if means.get('drug (paired)') is not None and means.get('swapped') is not None:
            # the arms that cover identical pairs: this is the clean comparison
            line['paired - swapped'] = f"{means['drug (paired)'] - means['swapped']:+.3f}"
        rows.append(line)

if rows:
    order = (['model', 'level', 'n', 'chance']
             + [n for n, _g, _c, _f in ARMS] + ['drug - swapped', 'paired - swapped'])
    frame = pd.DataFrame(rows)
    frame = frame[[c for c in order if c in frame.columns]]
    print(frame.to_string(index=False))
    frame.to_csv(f'{WORK}/drug_sites_comparison.csv', index=False)
    print(f'\nSaved -> {WORK}/drug_sites_comparison.csv')
    print('\nprecision@' + K + ', mean ± sd over seeds. "n" is the scorable pairs in that '
          'level (section 7), and "chance" is this ground truth\'s own chance level: a drug '
          'touches ~19 residues, so chance here is higher than against UniProt sites and '
          'lower than against the whole 85-residue pocket.')
    print('The answer is the last column: what is left after another drug\'s contacts in '
          'the same pocket have had their turn.')


## 10. Take the results with you

In [ ]:
ZIP = f'{WORK}/drug_sites_{DATASET}.zip'
FOLDERS = ' '.join(os.path.basename(f) for _n, _g, _c, f in ARMS)
!rm -f {ZIP}
!cd {WORK} && zip -qr {ZIP} {FOLDERS} drug_sites_comparison.csv && zip -qj {ZIP} drugsites_gpu0.log drugsites_gpu1.log 2>/dev/null
print(f'{ZIP}  {os.path.getsize(ZIP)/1e6:.2f} MB')
!unzip -l {ZIP} | tail -3
print(f'\nelapsed {(time.time() - START)/3600:.2f} h')
